In [1]:
import os
import pandas as pd
import wrds
from dotenv import load_dotenv


In [2]:
load_dotenv()
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
START_DATE, END_DATE = "2000-01-01", "2025-12-31"
START_YEAR, END_YEAR = 2000, 2025

UNIVERSE_FILTER = """
    sharetype = 'NS'
    AND securitytype = 'EQTY'
    AND securitysubtype = 'COM'
    AND usincflg = 'Y'
    AND issuertype IN ('ACOR', 'CORP')
    AND primaryexch IN ('N', 'Q', 'A')
    AND conditionaltype = 'RW'
"""

# query to restrict metadata to our universe
UNIVERSE_PERMNOS = f"SELECT DISTINCT permno FROM crsp.stksecurityinfohist WHERE {UNIVERSE_FILTER}"

wrap queries and download operations as functions

In [6]:
#
def connect_wrds():
    """ will fail if no .env file found or values missing"""
    os.environ["PGPASSWORD"] = os.getenv("WRDS_SECRET")
    return wrds.Connection(wrds_username=os.getenv("WRDS_USER"))


def fetch_compustat(conn):
    print("fetching Compustat Fundamentals (Annual) with CRSP link...")
    sql = f"""
        SELECT
            a.*,
            b.lpermno AS permno
        FROM comp.funda a
        JOIN crsp.ccmxpf_lnkhist b ON a.gvkey = b.gvkey
        WHERE b.linktype IN ('LU', 'LC') AND b.linkprim IN ('P', 'C')
          AND a.datadate BETWEEN '{START_DATE}' AND '{END_DATE}'
          AND a.datadate >= b.linkdt AND (a.datadate <= b.linkenddt OR b.linkenddt IS NULL)
          AND a.indfmt='INDL' AND a.datafmt='STD' AND a.popsrc='D' AND a.consol='C'
    """
    df = conn.raw_sql(sql, date_cols=["datadate"])
    df.to_parquet(f"{DATA_DIR}/compustat_funda.parquet", index=False)


def fetch_metadata(conn):
    queries = {
        "crsp_security_info": f"SELECT * FROM crsp.stksecurityinfohist WHERE {UNIVERSE_FILTER}",
        "crsp_delist": f"SELECT * FROM crsp.stkdelists WHERE permno IN ({UNIVERSE_PERMNOS})",
        "crsp_distributions": f"SELECT * FROM crsp.stkdistributions WHERE permno IN ({UNIVERSE_PERMNOS}) AND disexdt BETWEEN '{START_DATE}' AND '{END_DATE}'",
        "crsp_sp500_members": f"SELECT * FROM crsp.dsp500list_v2 WHERE mbrstartdt <= '{END_DATE}'",
        "crsp_market_index": f"SELECT date, vwretd, vwretx, ewretd, ewretx, sprtrn, totval, usdval FROM crsp.dsi WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}'",
        "ff_factors_daily": f"SELECT date, mktrf, smb, hml, rf, umd FROM ff.factors_daily WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}'"
    }
    
    for name, sql in queries.items():
        print(f"fetching {name}...")
        df = conn.raw_sql(sql)
        df.to_parquet(f"{DATA_DIR}/{name}.parquet", index=False)


def fetch_dsf_batched(conn):
    """ itereates over each calendar year downloads as parquet"""
    # Base directory for the partitioned dataset
    dsf_dir = f"{DATA_DIR}/dsf"
    os.makedirs(dsf_dir, exist_ok=True)

    for year in range(START_YEAR, END_YEAR + 1):
        print(f"fetching DSF for {year}...")
        sql = f"""
            SELECT * FROM crsp.dsf_v2 
            WHERE {UNIVERSE_FILTER} 
              AND dlycaldt BETWEEN '{year}-01-01' AND '{year}-12-31' 
        """
        df = conn.raw_sql(sql, date_cols=["dlycaldt"])
        
        if not df.empty:
            # hive style partition directory to read files later
            partition_dir = f"{dsf_dir}/year={year}"
            os.makedirs(partition_dir, exist_ok=True)
            
            # pandas to parquet file inside that directory
            out_path = f"{partition_dir}/data.parquet"
            df.to_parquet(out_path, index=False)

Create connection object to postgres DB, trigger each download script, then close connection.

In [ ]:

db = connect_wrds()
fetch_metadata(db)
fetch_compustat(db)
fetch_dsf_batched(db)
db.close()
print("Download complete")


Loading library list...
Done
fetching crsp_security_info...
fetching crsp_delist...
fetching crsp_distributions...
fetching crsp_sp500_members...
fetching crsp_market_index...
fetching ff_factors_daily...
fetching Compustat Fundamentals (Annual) with CRSP link...
